In [1]:
import os
import sys
from dotenv import load_dotenv

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"

    from google.colab import userdata
    DB_PASSWORD = userdata.get('DB_PASSWORD')
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

    load_dotenv()
    DB_PASSWORD = os.getenv("DB_PASSWORD")

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 947, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 947 (delta 52), reused 31 (delta 19), pack-reused 866 (from 1)
Receiving objects: 100% (947/947), 832.12 KiB | 12.24 MiB/s, done.
Resolving deltas: 100% (651/651), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:

from datetime import datetime, timedelta
import pytz

# Bangkok timezone
tz = pytz.timezone("Asia/Bangkok")

today = datetime.now(tz)

# shift back 10 days
shifted = today - timedelta(days=10)

YEAR = shifted.year
MONTH = shifted.month

print(YEAR, MONTH)

2026 4


In [3]:
import os
import pandas as pd

def load_csv_folder(folder):
    """
    Load all CSV files from a folder into a dictionary of DataFrames.

    Parameters
    ----------
    folder : str or Path
        Folder path containing CSV files.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary where key = filename, value = DataFrame.
    """

    data = {}

    folder = str(folder)  # ensures Path() also works

    for file in os.listdir(folder):
        if file.lower().endswith(".csv"):
            path = os.path.join(folder, file)

            df = pd.read_csv(
                path,
                dtype={
                    "BCODE": "string",
                    "MOBILE": "string",
                    "BILLNO": "string",
                },
                encoding="utf-8-sig",
                low_memory=False
            )

            data[file] = df
            print(f"Loaded: {file} -> {df.shape}")

    return data

In [4]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "02_staging",
    "VAT_Purchases",
    f"{YEAR}_{MONTH:02d}"
)

data = load_csv_folder(folder)

Loaded: VAT_PURCHASE_2026_04_BOOK_1_0.csv -> (170, 13)
Loaded: VAT_PURCHASE_2026_04_BOOK_2_0.csv -> (28, 13)
Loaded: VAT_PURCHASE_2026_04_BOOK_5_0.csv -> (8, 13)


In [5]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "03_curated",
)

data_raw = load_csv_folder(folder)

Loaded: shopee.csv -> (2733, 2)
Loaded: lazada.csv -> (3357, 2)
Loaded: tiktok.csv -> (980, 2)
Loaded: dim_date.csv -> (1827, 8)
Loaded: dim_category.csv -> (37, 2)
Loaded: dim_account.csv -> (2376, 4)
Loaded: dim_branch.csv -> (2, 3)
Loaded: dim_billtype.csv -> (9, 2)
Loaded: dim_product.csv -> (29507, 6)
Loaded: fact_sales_all.csv -> (775565, 37)
Loaded: fact_sales_bills_all.csv -> (291105, 49)


In [6]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter

def export_vat_report(
    df,
    sheet_name,
    title_name,
    wb=None,
    company_name="บริษัท เกียรติชัยอะไหล่ยนต์ 2007 จำกัด (สำนักงานใหญ่)",
    company_address="305 หมู่ 1 ต.ชุมแสง อ.วังจันทร์ จ.ระยอง 21210",
    tax_id="0215560000262",
    overwrite_sheet=True,   # if sheet exists, replace it
):
    # ✅ Safe df check (NO "if df:" allowed)
    if df is None:
        raise ValueError("df is None")
    if isinstance(df, pd.Series):
        raise ValueError("df must be a DataFrame, not a Series")
    if df.empty:
        # still create sheet with headers/title, or just skip
        pass

    # Workbook handling
    if wb is None:
        wb = Workbook()
        # remove default sheet if empty default
        if wb.sheetnames == ["Sheet"]:
            wb.remove(wb["Sheet"])

    # Handle existing sheet name
    if sheet_name in wb.sheetnames:
        if overwrite_sheet:
            wb.remove(wb[sheet_name])
        else:
            # make a unique name
            i = 2
            base = sheet_name
            while f"{base}_{i}" in wb.sheetnames:
                i += 1
            sheet_name = f"{base}_{i}"

    ws = wb.create_sheet(sheet_name)

    # Styles
    bold = Font(bold=True)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left = Alignment(horizontal="left", vertical="center", wrap_text=True)
    header_fill = PatternFill(start_color="F3DFD2", fill_type="solid")
    thin = Side(style="thin")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Title / Header area
    ws.merge_cells("A1:H1")
    ws["A1"] = title_name
    ws["A1"].font = Font(bold=True, size=14)
    ws["A1"].alignment = center

    ws.merge_cells("A3:C3"); ws["A3"] = "ชื่อสถานประกอบกิจการ"
    ws.merge_cells("D3:F3"); ws["D3"] = company_name
    ws.merge_cells("A4:C4"); ws["A4"] = "ที่อยู่สถานประกอบกิจการ"
    ws.merge_cells("D4:F4"); ws["D4"] = company_address
    ws.merge_cells("A5:C5"); ws["A5"] = "เลขประจำตัวผู้เสียภาษี"
    ws.merge_cells("D5:F5"); ws["D5"] = tax_id

    for r in [3, 4, 5]:
        ws[f"A{r}"].font = bold
        ws[f"A{r}"].alignment = left
        ws[f"D{r}"].alignment = left

    # Table start
    start_row = 7

    # Write df (header + rows)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start_row):
        for c_idx, value in enumerate(row, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            cell.border = border
            if r_idx == start_row:
                cell.font = bold
                cell.fill = header_fill
                cell.alignment = center
            else:
                cell.alignment = left

    ws.freeze_panes = f"A{start_row+1}"

    # Column widths (safe)
    for c_idx, col_name in enumerate(df.columns, 1):
        series = df[col_name].astype(str).fillna("")
        max_len = max(series.map(len).max(), len(str(col_name)))
        ws.column_dimensions[get_column_letter(c_idx)].width = min(max_len + 2, 60)

    # Money formatting
    money_like = {"มูลค่าสินค้า", "ภาษีมูลค่าเพิ่ม", "ยอดสุทธิ", "AMOUNT", "VAT", "TOTAL"}
    for c_idx, col_name in enumerate(df.columns, 1):
        name = str(col_name).strip()
        if name in money_like or any(k in name.upper() for k in ["AMOUNT", "VAT", "TOTAL"]):
            for r in range(start_row + 1, start_row + len(df) + 1):
                ws.cell(row=r, column=c_idx).number_format = '#,##0.00'

    date_cols = {"วันที่", "BILLDATE"}
    for c_idx, col_name in enumerate(df.columns, 1):
        if str(col_name).strip() in date_cols:
            for r in range(start_row + 1, start_row + len(df) + 1):
                cell = ws.cell(row=r, column=c_idx)
                cell.number_format = "dd/mm/yyyy"   # <-- your short format

    return wb

In [7]:
def clean_excel_text(df):
    return df.apply(
        lambda col: col.str.replace(r'^="|"$', '', regex=True)
        if col.dtype == "object" or str(col.dtype) == "string"
        else col
    )

In [8]:
import pandas as pd

def build_vat_report(
    df,
    rename_map=None,
    column_order=None,
    wb=None,
    sheet_name="REPORT",
    title_name="รายงานภาษี",
    clean_text=True,
    reset_seq=True,
    seq_col="SEQ",
    seq_out_col="ลำดับ",
    add_total_row=True,
    total_sum_cols=("มูลค่าสินค้า", "ภาษีมูลค่าเพิ่ม", "ยอดสุทธิ"),
    total_label_col="เลขที่",
    total_label_text="รวม",
    empty_action="write_headers",   # "write_headers" | "skip"
    **export_kwargs
):
    if df is None:
        raise ValueError("df is None")
    if isinstance(df, pd.Series):
        raise ValueError("df must be a DataFrame, not a Series")

    df = df.copy()

    if clean_text:
        df = clean_excel_text(df)

    # rename ONLY if rename_map is given AND it seems applicable
    if rename_map:
        if any(k in df.columns for k in rename_map.keys()):
            df = df.rename(columns=rename_map)

    # ---- build a safe sort key for วันที่ ----
    if "วันที่" in df.columns:
        raw_date = df["วันที่"].astype("string").str.strip()

        # try normal parse first
        sort_date = pd.to_datetime(raw_date, errors="coerce")

        # fallback for Thai BE format dd/mm/yyyy such as 09/04/2569
        thai_mask = sort_date.isna() & raw_date.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)
        if thai_mask.any():
            parts = raw_date[thai_mask].str.extract(r"^(?P<d>\d{2})/(?P<m>\d{2})/(?P<y>\d{4})$")
            y = pd.to_numeric(parts["y"], errors="coerce") - 543
            fallback_date = pd.to_datetime(
                parts["d"] + "/" + parts["m"] + "/" + y.astype("Int64").astype("string"),
                format="%d/%m/%Y",
                errors="coerce"
            )
            sort_date.loc[thai_mask] = fallback_date

        df["_sort_date"] = sort_date

    # sort by parsed date + bill no if available
    sort_cols = []
    if "_sort_date" in df.columns:
        sort_cols.append("_sort_date")
    if "เลขที่" in df.columns:
        sort_cols.append("เลขที่")

    if sort_cols:
        df = df.sort_values(sort_cols, kind="stable").reset_index(drop=True)

    # format Thai display date only after sorting
    if "วันที่" in df.columns:
        if "_sort_date" in df.columns:
            valid_mask = df["_sort_date"].notna()

            thai_display = (
                df.loc[valid_mask, "_sort_date"].dt.day.astype("Int64").astype("string").str.zfill(2)
                + "/"
                + df.loc[valid_mask, "_sort_date"].dt.month.astype("Int64").astype("string").str.zfill(2)
                + "/"
                + (df.loc[valid_mask, "_sort_date"].dt.year + 543).astype("Int64").astype("string")
            )

            original_date = df["วันที่"].astype("string").copy()
            original_date.loc[valid_mask] = thai_display
            df["วันที่"] = original_date
        else:
            df["วันที่"] = df["วันที่"].astype("string")

    # reset seq (supports either SEQ or ลำดับ)
    if reset_seq:
        df = df.reset_index(drop=True)
        if seq_col in df.columns:
            df[seq_col] = range(1, len(df) + 1)
        elif seq_out_col in df.columns:
            df[seq_out_col] = range(1, len(df) + 1)
        else:
            df[seq_out_col] = range(1, len(df) + 1)

    # limit target column to last 13 characters
    if total_label_col in df.columns:
        df[total_label_col] = (
            df[total_label_col]
            .astype("string")
            .str[-13:]
        )

    # remove helper col before empty/export
    if "_sort_date" in df.columns:
        df = df.drop(columns=["_sort_date"])

    # empty df handling
    if df.empty:
        return df, wb

    # reorder if provided
    if column_order:
        missing = [c for c in column_order if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns for column_order: {missing}")
        df = df.loc[:, list(column_order)]

    # add total row
    if add_total_row:
        total_row = {c: "" for c in df.columns}

        for c in total_sum_cols:
            if c in df.columns:
                total_row[c] = df[c].sum()

        if total_label_col in df.columns:
            total_row[total_label_col] = total_label_text

        if seq_col in total_row:
            total_row[seq_col] = ""
        if seq_out_col in total_row:
            total_row[seq_out_col] = ""

        df = pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)

    wb = export_vat_report(
        df,
        sheet_name=sheet_name,
        title_name=title_name,
        wb=wb,
        **export_kwargs
    )

    return df, wb

In [9]:
def get_df(data, key):
    return data[key].copy() if key in data else pd.DataFrame()

df_book_1 = get_df(data, f"VAT_PURCHASE_{YEAR}_{MONTH:02d}_BOOK_1_0.csv")
df_book_2 = get_df(data, f"VAT_PURCHASE_{YEAR}_{MONTH:02d}_BOOK_2_0.csv")
df_book_5 = get_df(data, f"VAT_PURCHASE_{YEAR}_{MONTH:02d}_BOOK_5_0.csv")
df_book_6 = get_df(data, f"VAT_PURCHASE_{YEAR}_{MONTH:02d}_BOOK_6_0.csv")
df_unknown = get_df(data, f"VAT_PURCHASE_{YEAR}_{MONTH:02d}_BOOK_UNKNOWN.csv")

In [10]:
wb = None

In [11]:
column_order = [
    "ลำดับ",
    "วันที่",
    "เลขที่",
    "ชื่อผู้ขายสินค้า",
    "เลขผู้เสียภาษี",
    "มูลค่าสินค้า",
    "ภาษีมูลค่าเพิ่ม",
    "ยอดสุทธิ",
    "รายการสินค้า",
    "หมายเหตุ"
]

In [12]:
rename_map = {
    "SEQ": "ลำดับ",
    "BILLDATE": "วันที่",
    "BILLNO": "เลขที่",
    "ACCTNAME": "ชื่อผู้ขายสินค้า",
    "REMARKS_EXTRACT": "เลขผู้เสียภาษี",
    "BEFORETAX": "มูลค่าสินค้า",
    "TAX": "ภาษีมูลค่าเพิ่ม",
    "AFTERTAX": "ยอดสุทธิ",
    "DETAIL": "รายการสินค้า",
    "PO": "หมายเหตุ"
}

df_book_1_report, wb = build_vat_report(
    df_book_1,
    rename_map,
    column_order,
    wb=wb,
    sheet_name="เครดิต",
    title_name="รายงานภาษีซื้อ"
)

df_book_2_report, wb = build_vat_report(
    df_book_2,
    rename_map,
    column_order,
    wb=wb,
    sheet_name="สด",
    title_name="รายงานภาษีซื้อ"
)

df_book_5_report, wb = build_vat_report(
    df_book_5,
    rename_map,
    column_order,
    wb=wb,
    sheet_name="ลดหนี้ซื้อ",
    title_name="รายงานภาษีซื้อ"
)

df_book_6_report, wb = build_vat_report(
    df_book_6,
    rename_map,
    column_order,
    wb=wb,
    sheet_name="เพิ่มหนี้ซื้อ",
    title_name="รายงานภาษีซื้อ"
)

df_unknown_report, wb = build_vat_report(
    df_unknown,
    rename_map,
    column_order,
    wb=wb,
    sheet_name="Unknown",
    title_name="รายงานภาษีซื้อ"
)


In [13]:
from sqlalchemy import create_engine

DATABASE_URL = (
    "postgresql+psycopg2://"
    f"postgres.jdzitzsucntqbjvwiwxm:{DB_PASSWORD}@"
    "aws-0-ap-southeast-1.pooler.supabase.com:5432/postgres"
)

engine_supabase = create_engine(DATABASE_URL)

# test connection
with engine_supabase.connect() as conn_test:
    print("Connected via SQLAlchemy ✅")

Connected via SQLAlchemy ✅


In [14]:
df_expense = pd.read_sql(
    f"""
    SELECT *
    FROM vw_expense_entry_flat_tax
    WHERE created_at >= DATE '{YEAR}-{MONTH:02d}-10'
      AND created_at <=  (DATE '{YEAR}-{MONTH:02d}-10' + INTERVAL '1 month');
    """,
    engine_supabase
)

In [15]:
df_expense.columns

Index(['entry_uuid', 'receipt_uuid', 'receipt_number', 'doc_type',
       'ref_receipt_uuid', 'ref_receipt_number', 'receipt_date', 'receipt_day',
       'receipt_month_start', 'receipt_month_year', 'receipt_year',
       'receipt_month', 'user_id', 'branch_uuid', 'branch_name',
       'payment_uuid', 'payment_method_name', 'party_uuid', 'party_name',
       'tax_payer_id', 'party_legal_name', 'party_tax_address', 'item_uuid',
       'item_name', 'category_uuid', 'category_name', 'entry_detail',
       'quantity', 'unit_price', 'entry_discount', 'entry_amount', 'sign',
       'signed_entry_amount', 'signed_gross_amount', 'signed_entry_discount',
       'total_amount', 'signed_total', 'receipt_discount', 'vat',
       'withholding', 'tax_exempt', 'voucher_description', 'receipt_remark',
       'has_images', 'created_at'],
      dtype='object')

In [16]:
import pandas as pd

def summarize_expense_receipts(df_expense_vat: pd.DataFrame) -> pd.DataFrame:
    df = df_expense_vat.copy()

    # ensure numeric
    df["signed_entry_amount"] = pd.to_numeric(df["signed_entry_amount"], errors="coerce").fillna(0)
    df["signed_total"] = pd.to_numeric(df["signed_total"], errors="coerce")

    # pick the row (per receipt_uuid) with biggest signed_entry_amount
    idx_max = df.groupby("receipt_uuid")["signed_entry_amount"].idxmax()
    df_pick = df.loc[idx_max, ["receipt_uuid", "entry_detail"]].copy()

    # receipt-level aggregation
    agg = (
        df.groupby("receipt_uuid", as_index=False)
          .agg({
              "receipt_date": "first",
              "receipt_number": "first",
              "party_name": "first",
              "tax_payer_id": "first",
              "signed_entry_amount": "sum",   # sum lines
              "signed_total": "first",        # header measure (don’t sum)
              "doc_type": "first",
              "branch_uuid": "first",
              "ref_receipt_number": "first"
          })
    )

    # merge the picked entry_detail
    out = agg.merge(df_pick, on="receipt_uuid", how="left")

    # final column order
    out = out[[
        "receipt_date",
        "receipt_number",
        "party_name",
        "tax_payer_id",
        "entry_detail",
        "signed_entry_amount",
        "signed_total",
        "doc_type",
        "branch_uuid",
        "ref_receipt_number"
    ]]

    return out

In [17]:
df_expense_vat = df_expense[df_expense["vat"] != 0]

df_expense_vat = summarize_expense_receipts(df_expense_vat)

In [18]:
df_receipt = df_expense_vat[df_expense_vat["doc_type"] == "RECEIPT"].copy()
df_credit_note = df_expense_vat[df_expense_vat["doc_type"] == "CREDIT_NOTE"].copy()

In [19]:
df_receipt["vat_amount"] = df_receipt["signed_entry_amount"] * 0.07
df_receipt["after_vat"]  = df_receipt["signed_entry_amount"] * 1.07

df_credit_note["vat_amount"] = df_credit_note["signed_entry_amount"] * 0.07
df_credit_note["after_vat"]  = df_credit_note["signed_entry_amount"] * 1.07

In [20]:
HQ_UUID = "c93efb5f-07c9-4229-b6b3-568ce1c0a9ab"

df_receipt["branch_uuid"] = df_receipt["branch_uuid"].astype(str)
df_credit_note["branch_uuid"] = df_credit_note["branch_uuid"].astype(str)

df_receipt_hq = df_receipt.loc[df_receipt["branch_uuid"].eq(HQ_UUID)].copy()
df_receipt_syp = df_receipt.loc[~df_receipt["branch_uuid"].eq(HQ_UUID)].copy()

df_credit_note_hq = df_credit_note.loc[df_credit_note["branch_uuid"].eq(HQ_UUID)].copy()
df_credit_note_syp  = df_credit_note.loc[~df_credit_note["branch_uuid"].eq(HQ_UUID)].copy()


In [21]:
rename_map_expense = {
    "seq": "ลำดับ",
    "receipt_date": "วันที่",
    "receipt_number": "เลขที่",
    "party_name": "ชื่อผู้ขายสินค้า",
    "tax_payer_id": "เลขผู้เสียภาษี",
    "signed_entry_amount": "มูลค่าสินค้า",
    "vat_amount": "ภาษีมูลค่าเพิ่ม",
    "after_vat": "ยอดสุทธิ",
    "entry_detail": "รายการสินค้า",
    "ref_receipt_number": "หมายเหตุ",
}

column_order_expense = [
    "ลำดับ", "วันที่", "เลขที่", "ชื่อผู้ขายสินค้า", "เลขผู้เสียภาษี",
    "มูลค่าสินค้า", "ภาษีมูลค่าเพิ่ม", "ยอดสุทธิ", "รายการสินค้า", "หมายเหตุ"
]

df_credit_note_hq_report, wb = build_vat_report(
    df_credit_note_hq,
    rename_map_expense,
    column_order_expense,
    wb=wb,
    sheet_name="ลดหนี้ค่าใช้จ่าย",
    title_name="รายงานภาษีซื้อ (Expense Credit Note)"
)

df_receipt_hq_report, wb = build_vat_report(
    df_receipt_hq,
    rename_map_expense,
    column_order_expense,
    wb=wb,
    sheet_name="ค่าใช้จ่าย",
    title_name="รายงานภาษีซื้อ (Expense Receipt)"
)

wb_syp = None

df_credit_note_syp_report, wb_syp = build_vat_report(
    df_credit_note_syp,
    rename_map_expense,
    column_order_expense,
    wb=wb_syp,
    sheet_name="Expense Credit Note",
    title_name="รายงานภาษีซื้อ (Expense Credit Note)",
    company_name="บริษัท เกียรติชัยอะไหล่ยนต์ 2007 จำกัด (สาขาที่ 00003)",
    company_address="16/2 หมู่ 2 ต.ห้วยทับมอญ อ.เขาชะเมา จ.ระยอง 21110",
    tax_id="0215560000262",
)

df_receipt_syp_report, wb_syp = build_vat_report(
    df_receipt_syp,
    rename_map_expense,
    column_order_expense,
    wb=wb_syp,
    sheet_name="Expense Receipt",
    title_name="รายงานภาษีซื้อ (Expense Receipt)",
    company_name="บริษัท เกียรติชัยอะไหล่ยนต์ 2007 จำกัด (สาขาที่ 00003)",
    company_address="16/2 หมู่ 2 ต.ห้วยทับมอญ อ.เขาชะเมา จ.ระยอง 21110",
    tax_id="0215560000262",
)

In [22]:
def remove_total_row(df, label_col="เลขที่", label_text="รวม"):
    if label_col in df.columns:
        return df[df[label_col] != label_text].copy()
    return df.copy()

def strip_totals(dfs):
    return [remove_total_row(df) for df in dfs]

df_purchase_all_hq = pd.concat(strip_totals([
    df_book_1_report,
    df_book_2_report,
    df_book_5_report,
    df_book_6_report,
    df_unknown_report,
    df_receipt_hq_report,
    df_credit_note_hq_report,
]), ignore_index=True)

df_purchase_all_syp = pd.concat(strip_totals([
    df_receipt_syp_report,
    df_credit_note_syp_report,
]), ignore_index=True)

In [23]:
df_purchase_all_hq_report, wb = build_vat_report(
    df_purchase_all_hq,
    rename_map_expense,
    column_order_expense,
    wb=wb,
    sheet_name="รวม",
    title_name="รายงานภาษีซื้อ (รวม)",
    clean_text=False,   # ✅ important
)

/tmp/ipykernel_28223/3346214278.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sort_date = pd.to_datetime(raw_date, errors="coerce")


In [24]:
df_purchase_all_syp_report, wb_syp = build_vat_report(
    df_purchase_all_syp,
    rename_map_expense,
    column_order_expense,
    wb=wb_syp,
    sheet_name="รวม",
    title_name="รายงานภาษีซื้อ (รวม)",
    clean_text=False,   # ✅ important
    company_name="บริษัท เกียรติชัยอะไหล่ยนต์ 2007 จำกัด (สาขาที่ 00003)",
    company_address="16/2 หมู่ 2 ต.ห้วยทับมอญ อ.เขาชะเมา จ.ระยอง 21110",
    tax_id="0215560000262",
)

In [25]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "01_VAT_Purchases_Report",
    f"vat_purchases_report_{YEAR}_{MONTH:02}_hq.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

wb.save(folder)

In [27]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "01_VAT_Purchases_Report",
    f"vat_purchases_report_{YEAR}_{MONTH:02}_syp.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

if wb_syp is not None and len(wb_syp.sheetnames) > 0:
    wb_syp.save(folder)
else:
    print("No SYP VAT data. Skip workbook generation.")

No SYP VAT data. Skip workbook generation.
